# Notebook 2: Tokenization Explained
### From Raw Text to Training Tensors

**Series**: Fine-Tuning Flan-T5 for Dialogue Summarization (5-Part Series)  
**This notebook**: Part 2 of 5 — standalone, no other notebook required

---

## What You Will Learn

Before a language model can train on text, every word (and sub-word) must be converted
into a number. This notebook demystifies that process end-to-end.

By the end you will understand:

1. **What a tokenizer does**: text → token IDs → back to text
2. **Why Flan-T5 uses SentencePiece**: subword tokenization and why it matters
3. **The `"summarize: "` instruction prefix** and why it is essential for instruction-tuned models
4. **The `-100` label masking trick**: how padding tokens are excluded from the training loss
5. **The exact tensor shapes** that flow into the trainer

---

## Connection to the Codebase

Everything in this notebook maps directly to **`training/dataset.py`** —
specifically the `tokenize_function()` that prepares every batch the trainer sees:

```python
# training/dataset.py  (simplified excerpt)
def tokenize_function(examples, tokenizer):
    inputs = ["summarize: " + dialogue for dialogue in examples["dialogue"]]
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")
    labels = tokenizer(text_target=examples["summary"], max_length=128, truncation=True, padding="max_length")
    label_ids = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = label_ids
    return model_inputs
```

We will rebuild every line of that function from first principles.

In [ ]:
# Install required packages
# transformers 5.x removed the old pipeline API; we use AutoTokenizer directly
#!pip install -q "transformers>=5.2.0" "datasets>=4.6.0" torch

In [1]:
# Load the Flan-T5 tokenizer only — no model weights downloaded yet.
# The tokenizer alone is ~2 MB; the full model is ~900 MB.
# This keeps the notebook fast and Colab-friendly.
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

# --- Basic tokenization demonstration ---
text = "Hello, how are you doing today?"

# tokenize() returns human-readable subword tokens (strings, NOT yet numbers)
tokens = tokenizer.tokenize(text)

# encode() returns the integer IDs for each token
# Note: encode() also appends an end-of-sequence token (</s> = ID 1) automatically
ids = tokenizer.encode(text)

# decode() converts IDs back to a string — the round-trip
decoded = tokenizer.decode(ids)

print("Text:   ", text)
print("Tokens: ", tokens)
print("IDs:    ", ids)
print("Decoded:", decoded)

Text:    Hello, how are you doing today?
Tokens:  ['▁Hello', ',', '▁how', '▁are', '▁you', '▁doing', '▁today', '?']
IDs:     [8774, 6, 149, 33, 25, 692, 469, 58, 1]
Decoded: Hello, how are you doing today?</s>


## How SentencePiece Subword Tokenization Works

You may have noticed that the tokenizer split `"Hello"` into pieces like `▁Hello` —
the `▁` (underscore) is the SentencePiece marker for "this token starts after a space".

### Why subword tokenization?

Older approaches used **word-level** tokenization: every word is one token. The problem
is that rare words (e.g. `"unbelievably"`) just become `<UNK>` (unknown), and the model
learns nothing about them.

**SentencePiece** (used by Flan-T5, T5, LLaMA, and many others) splits words into
frequent sub-units:

```
"unbelievably"  →  ["▁un", "believ", "ably"]
```

This means:
- The vocabulary can stay small (~32 000 tokens for Flan-T5)
- The model still sees meaningful pieces of rare words
- The same vocabulary works across many languages

---

## The `"summarize: "` Instruction Prefix

Flan-T5 was **instruction-tuned**: it was trained on dozens of NLP tasks simultaneously,
where each task was identified by a natural language prefix:

| Task | Prefix |
|---|---|
| Summarization | `"summarize: "` |
| Translation | `"translate English to French: "` |
| Question answering | `"question: ... context: ..."` |
| Sentiment | `"Is this sentence positive or negative? "` |

Without the prefix, the model does not know which task you want it to perform.
With `"summarize: "`, you are speaking the model's training language.

This is why `training/dataset.py` does:

```python
inputs = ["summarize: " + dialogue for dialogue in examples["dialogue"]]
```

It is not cosmetic — it is the key that unlocks the model's summarization capability.

In [2]:
import random
random.seed(42)  # reproducibility for any random sampling below

# A real SAMSum-style dialogue (from the training set)
example_dialogue = (
    "Hannah: Hey, did you hear about the concert next Friday?\n"
    "Liam: No, what concert?\n"
    "Hannah: The local jazz band is playing at the park. Free admission!\n"
    "Liam: That sounds amazing. Should we go together?\n"
    "Hannah: Absolutely! Let's meet at the entrance at 7pm.\n"
    "Liam: Perfect. See you then!"
)

# --- Tokenize WITHOUT the prefix ---
without_prefix = tokenizer.tokenize(example_dialogue)

# --- Tokenize WITH the "summarize: " prefix ---
with_prefix = tokenizer.tokenize("summarize: " + example_dialogue)

print("=== Effect of the 'summarize: ' prefix ===")
print(f"Token count WITHOUT prefix: {len(without_prefix)}")
print(f"Token count WITH    prefix: {len(with_prefix)}")
print(f"Overhead: +{len(with_prefix) - len(without_prefix)} tokens")
print()
print("First 10 tokens WITHOUT prefix:", without_prefix[:10])
print("First 10 tokens WITH    prefix:", with_prefix[:10])
print()
# The prefix itself tokenizes into just a few pieces — small overhead, huge benefit
print("How does 'summarize: ' tokenize?")
print(tokenizer.tokenize("summarize: "))

=== Effect of the 'summarize: ' prefix ===
Token count WITHOUT prefix: 72
Token count WITH    prefix: 74
Overhead: +2 tokens

First 10 tokens WITHOUT prefix: ['▁Hannah', ':', '▁Hey', ',', '▁did', '▁you', '▁hear', '▁about', '▁the', '▁concert']
First 10 tokens WITH    prefix: ['▁summarize', ':', '▁Hannah', ':', '▁Hey', ',', '▁did', '▁you', '▁hear', '▁about']

How does 'summarize: ' tokenize?
['▁summarize', ':']


In [3]:
# Full input tokenization with padding to a fixed length
# This mirrors exactly what tokenize_function() in training/dataset.py does for inputs.

dialogue = "Amanda: I baked cookies. Jerry: Sounds great!"
prefixed = "summarize: " + dialogue

encoded = tokenizer(
    prefixed,
    max_length=512,       # same constant as INPUT_MAX_LENGTH in dataset.py
    truncation=True,      # cut sequences longer than max_length
    padding="max_length", # pad shorter sequences with pad_token_id to reach max_length
)

print("input_ids length:", len(encoded["input_ids"]))
print("First 20 token IDs:", encoded["input_ids"][:20])
print("Last 20 token IDs (should be padding):", encoded["input_ids"][-20:])
print("Pad token ID:", tokenizer.pad_token_id)

# Count how many tokens are real content vs. padding
non_pad = sum(1 for t in encoded["input_ids"] if t != tokenizer.pad_token_id)
print(f"\nNon-padding tokens: {non_pad} / {len(encoded['input_ids'])}")
print(f"Padding tokens:     {len(encoded['input_ids']) - non_pad} / {len(encoded['input_ids'])}")
print()

# The attention_mask marks which positions the model should attend to:
#   1 = real token  (model pays attention)
#   0 = padding     (model ignores)
print("attention_mask first 20:", encoded["attention_mask"][:20])
print("attention_mask last  20:", encoded["attention_mask"][-20:])

input_ids length: 512
First 20 token IDs: [21603, 10, 21542, 10, 27, 13635, 5081, 5, 16637, 10, 17112, 248, 55, 1, 0, 0, 0, 0, 0, 0]
Last 20 token IDs (should be padding): [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Pad token ID: 0

Non-padding tokens: 14 / 512
Padding tokens:     498 / 512

attention_mask first 20: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
attention_mask last  20: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [4]:
# Label tokenization and the -100 masking trick
#
# Labels are the *target* sequence the model must learn to generate.
# We pad them too (to batch uniformly), but padding positions must NOT
# contribute to the loss — otherwise the model wastes capacity learning
# to predict padding tokens.
#
# Solution: replace pad_token_id with -100.
# PyTorch's CrossEntropyLoss ignores positions where the target is -100.
# This is a built-in PyTorch convention, not a Hugging Face invention.

summary = "Amanda baked cookies. Jerry approves."

# text_target= tells the tokenizer this is a decoder-side (label) sequence.
# For T5-family models this matters because the encoder and decoder may use
# slightly different pre/post-processing.
label_enc = tokenizer(
    text_target=summary,
    max_length=128,       # same constant as LABEL_MAX_LENGTH in dataset.py
    truncation=True,
    padding="max_length",
)

raw_labels = label_enc["input_ids"]

# Replace every pad_token_id with -100 so CrossEntropyLoss skips it
masked_labels = [(t if t != tokenizer.pad_token_id else -100) for t in raw_labels]

print("Raw labels (first 20):   ", raw_labels[:20])
print("Masked labels (first 20):", masked_labels[:20])
print(f"\n-100 count: {masked_labels.count(-100)} padding tokens masked out")
print(f"Real label tokens: {len([t for t in masked_labels if t != -100])} / {len(masked_labels)}")
print()
print("Why -100? PyTorch CrossEntropyLoss ignores index=-100 by default.")
print("Concretely: loss = CrossEntropyLoss(ignore_index=-100)(logits, labels)")
print("Padding tokens contribute zero gradient — the model is not penalised for them.")

Raw labels (first 20):    [21542, 13635, 5081, 5, 16637, 15444, 7, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Masked labels (first 20): [21542, 13635, 5081, 5, 16637, 15444, 7, 5, 1, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]

-100 count: 119 padding tokens masked out
Real label tokens: 9 / 128

Why -100? PyTorch CrossEntropyLoss ignores index=-100 by default.
Concretely: loss = CrossEntropyLoss(ignore_index=-100)(logits, labels)
Padding tokens contribute zero gradient — the model is not penalised for them.


In [7]:
import transformers
print(transformers.__version__)  # needs to be >= 4.21.0

5.2.0


In [10]:
def tokenize_function(examples, tokenizer):
    inputs = ["summarize: " + d for d in examples["dialogue"]]
    
    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length",
    )
    
    labels = tokenizer(
        text_target=list(examples["summary"]),  # ← cast to plain list
        max_length=128,
        truncation=True,
        padding="max_length",
    )
    
    label_ids = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = label_ids
    return model_inputs

In [11]:
print(type(batch_dict["summary"]))
print(type(batch_dict["summary"][0]))

<class 'datasets.arrow_dataset.Column'>
<class 'str'>


In [12]:
# Put it all together: reproduce tokenize_function() from training/dataset.py
# and run it on a real batch of 3 SAMSum examples.

from datasets import load_dataset

# Load only the test split to keep download time minimal (~1 MB vs ~10 MB for train)
samsum = load_dataset("knkarthick/samsum", split="test")

random.seed(42)
# Pick 3 deterministic examples from the test split
indices = random.sample(range(len(samsum)), 3)
batch = samsum.select(indices)

print("=== 3 raw SAMSum examples ===")
for i, example in enumerate(batch):
    print(f"\n--- Example {i+1} ---")
    print("Dialogue (truncated):", example["dialogue"][:120], "...")
    print("Summary:             ", example["summary"])

print("\n" + "="*60)
print("=== Running tokenize_function() on the batch ===")
print("="*60)

# ----- exact copy of tokenize_function from training/dataset.py -----
def tokenize_function(examples, tokenizer):
    """Tokenize inputs and labels for seq2seq training."""
    # Prepend task prefix so Flan-T5 knows what we want
    inputs = ["summarize: " + dialogue for dialogue in examples["dialogue"]]

    # Encode the (prefixed) input dialogues
    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length",
    )

    # Encode the target summaries as decoder labels
    labels = tokenizer(
    text_target=list(examples["summary"]),  # ← list() not list[Any]()
    max_length=128,
    truncation=True,
    padding="max_length",
    )

    # Replace padding token id in labels with -100 so it's ignored in loss
    label_ids = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = label_ids
    return model_inputs
# -------------------------------------------------------------------

# Convert the batch HuggingFace Dataset to a plain dict (same type the real
# dataset.map() lambda passes in when batched=True)
batch_dict = {col: batch[col] for col in batch.column_names}

result = tokenize_function(batch_dict, tokenizer)

# Inspect the output shapes
print(f"\nKeys in result:       {list(result.keys())}")
print(f"input_ids shape:      ({len(result['input_ids'])} examples, {len(result['input_ids'][0])} tokens each)")
print(f"attention_mask shape: ({len(result['attention_mask'])} examples, {len(result['attention_mask'][0])} tokens each)")
print(f"labels shape:         ({len(result['labels'])} examples, {len(result['labels'][0])} tokens each)")

print("\n--- Example 1: first 20 input_ids ---")
print(result["input_ids"][0][:20])

print("\n--- Example 1: first 20 labels (note -100 for padding) ---")
print(result["labels"][0][:20])

# Verify the -100 masking worked correctly
for i in range(3):
    n_masked = result["labels"][i].count(-100)
    n_real = len(result["labels"][i]) - n_masked
    print(f"\nExample {i+1}: {n_real} real label tokens, {n_masked} padding tokens masked to -100")

=== 3 raw SAMSum examples ===

--- Example 1 ---
Dialogue (truncated): Richie: Pogba
Clay: Pogboom
Richie: what a s strike yoh!
Clay: was off the seat the moment he chopped the ball back to h ...
Summary:              Richie and Clay saw a very good football game, with one football player chopping the ball back to his foot, which was particularly exciting. Jose has trust in that player. 

--- Example 2 ---
Dialogue (truncated): Lincoln: Heeyyy ;* whats up
Fatima: I talked to Jenson, he’s not too happy ;p
Lincoln: the place sucks??
Fatima: No, the ...
Summary:              Fatima is worried about Jenson and Alene. Alene has issues. Lincoln doesn't want Fatima to worry about others too much.

--- Example 3 ---
Dialogue (truncated): Ollie: Okay, Kelly! Ur up nxt!
Kelly: Me? I don't wanna.
Mickey: C'mon!
Jessica: Yeah! What's yours?
Kelly: Fine. It's a ...
Summary:              Kelly is scared of sculpture garden figures in Finnland, she finds figure's faces morbid. For Ollie it's Nagoro v

## Summary: What You Have Seen

### Connecting back to the codebase

| What we explored | Where it lives |
|---|---|
| SentencePiece tokenization | `AutoTokenizer.from_pretrained("google/flan-t5-base")` |
| `"summarize: "` prefix | `training/dataset.py` → `tokenize_function()`, line 3 |
| `max_length=512`, `padding="max_length"` | `training/dataset.py` → `INPUT_MAX_LENGTH = 512` |
| `text_target=` for labels | `training/dataset.py` → `tokenize_function()`, label block |
| `-100` masking | `training/dataset.py` → `tokenize_function()`, `label_ids` list-comp |
| Three output tensors | `input_ids`, `attention_mask`, `labels` |

### The three tensors the trainer receives

```
input_ids      shape: (batch, 512)   — prefixed & padded dialogue token IDs
attention_mask shape: (batch, 512)   — 1 for real tokens, 0 for padding
labels         shape: (batch, 128)   — summary token IDs; padding replaced by -100
```

### What `tokenize_function()` in `training/dataset.py` does

The function we reproduced in Cell 8 is **exactly** what runs inside
`dataset.map(lambda examples: tokenize_function(examples, tokenizer), batched=True)`.
HuggingFace `datasets` calls it once per shard of the dataset, passing
a dict of lists — the same `batch_dict` structure we used above.

### What happens next

These tensors are consumed by **`Seq2SeqTrainer`** in `training/train.py`:

```python
# training/train.py  (excerpt)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],     # <-- the tensors you just built
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
)
trainer.train()
```

In **Notebook 3 (`training_demo.ipynb`)** you will see the trainer consume
these tensors directly, watch the loss curve drop, and understand what
"one training step" actually computes.